# EfficientNet-TrOCR Training Notebook

Optimized for RTX 4000 Ada. No data augmentation.

In [ ]:
!pip install --upgrade transformers datasets jiwer evaluate Levenshtein huggingface_hub tensorboard tqdm

In [ ]:
# --- Imports ---
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from PIL import Image
from transformers import (
    VisionEncoderDecoderModel, TrOCRProcessor, BertTokenizer,
    AutoImageProcessor, EfficientNetModel, get_linear_schedule_with_warmup
)
from transformers.modeling_outputs import BaseModelOutput
from huggingface_hub import login

# Enable mixed precision
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

In [ ]:
# --- Set GPU/CPU Target ---
GPU = 'rtx4000'  # Options: 'rtx4000', 'p100', '2200g', 'rx580', 't4x2'

# --- Optimization settings for each device ---
GPU_CONFIGS = {
    'rtx4000': {
        'BATCH_SIZE': 16,
        'NUM_WORKERS': 4,
        'ENCODER_LR': 1e-5,
        'DECODER_LR': 5e-5,
        'FREEZE_ENCODER_EPOCHS': 3,
        'USE_TORCH_COMPILE': True,
        'AMP': True
    },
    'p100': {
        'BATCH_SIZE': 8,
        'NUM_WORKERS': 2,
        'ENCODER_LR': 5e-6,
        'DECODER_LR': 2e-5,
        'FREEZE_ENCODER_EPOCHS': 2,
        'USE_TORCH_COMPILE': False,
        'AMP': True
    },
    '2200g': {  # AMD Ryzen 3 2200G CPU (no CUDA)
        'BATCH_SIZE': 2,
        'NUM_WORKERS': 0,
        'ENCODER_LR': 1e-6,
        'DECODER_LR': 5e-6,
        'FREEZE_ENCODER_EPOCHS': 1,
        'USE_TORCH_COMPILE': False,
        'AMP': False
    },
    'rx580': {  # AMD RX580 2048SP (assume ROCm or CPU fallback)
        'BATCH_SIZE': 4,
        'NUM_WORKERS': 1,
        'ENCODER_LR': 2e-6,
        'DECODER_LR': 1e-5,
        'FREEZE_ENCODER_EPOCHS': 1,
        'USE_TORCH_COMPILE': False,
        'AMP': False
    },
    't4x2': {  # Kaggle T4 x2 (2x NVIDIA T4)
        'BATCH_SIZE': 24,
        'NUM_WORKERS': 6,
        'ENCODER_LR': 1.5e-5,
        'DECODER_LR': 7e-5,
        'FREEZE_ENCODER_EPOCHS': 3,
        'USE_TORCH_COMPILE': True,
        'AMP': True
    }
}

assert GPU in GPU_CONFIGS, f"Unknown GPU config: {GPU}"
cfg = GPU_CONFIGS[GPU]

BATCH_SIZE = cfg['BATCH_SIZE']
NUM_WORKERS = cfg['NUM_WORKERS']
ENCODER_LR = cfg['ENCODER_LR']
DECODER_LR = cfg['DECODER_LR']
FREEZE_ENCODER_EPOCHS = cfg['FREEZE_ENCODER_EPOCHS']
USE_TORCH_COMPILE = cfg['USE_TORCH_COMPILE']
AMP = cfg['AMP']

In [ ]:
# --- Configuration ---
MODEL_SOURCE_PATH = "ds1219/pandithv2"
EFFICIENTNET_MODEL_NAME = "google/efficientnet-b0"
SAVE_MODEL_PATH = "./my_combined_trocr_efficientnet_model"
NUM_EPOCHS = 5

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print(f"GPU config: {GPU}")

In [ ]:
# --- HuggingFace Login (if needed) ---
# access_token = user_secrets.get_secret("HF_Token")
# login(token=access_token)

## Dataset Definition

In [ ]:
class OCRDatasetFromTSV(Dataset):
    """Custom OCR dataset that reads image paths and labels from a .tsv file."""
    def __init__(self, tsv_path, images_dir, image_processor, tokenizer, max_length=128):
        df = pd.read_csv(tsv_path, sep='\t')
        self.image_paths = [f"{images_dir}/{fname}" for fname in df["file"].tolist()]
        self.texts = df["labels"].tolist()
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        pixel_values = self.image_processor(images=image, return_tensors="pt").pixel_values.squeeze()
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        labels = encoding.input_ids.squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

## Model Definition

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        query = self.query(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.key(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.value(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        attention_scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        attended_values = torch.matmul(attention_weights, value)
        attended_values = attended_values.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        output = self.out_proj(attended_values)
        return output

class EfficientNetCompatibleWrapper(nn.Module):
    def __init__(self, model_name="google/efficientnet-b0", target_hidden_size=768, num_attention_heads=8, dropout=0.1):
        super().__init__()
        self.model = EfficientNetModel.from_pretrained(model_name)
        dummy_input = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            dummy_output = self.model(pixel_values=dummy_input)
            self.efficientnet_hidden_size = dummy_output.last_hidden_state.shape[1]
        self.target_hidden_size = target_hidden_size
        self.attention = SelfAttention(embed_dim=self.efficientnet_hidden_size, num_heads=num_attention_heads, dropout=dropout)
        self.projection = nn.Linear(self.efficientnet_hidden_size, self.target_hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.model.config.hidden_size = target_hidden_size
        self.model.config.cross_attention_hidden_size = None
        self.config = self.model.config
        self.main_input_name = "pixel_values"
    def get_output_embeddings(self):
        return None
    def forward(self, pixel_values, output_attentions=None, output_hidden_states=None, return_dict=True, **kwargs):
        efficientnet_forward_kwargs = {"pixel_values": pixel_values}
        if output_hidden_states is not None:
            efficientnet_forward_kwargs["output_hidden_states"] = output_hidden_states
        if return_dict is not None:
            efficientnet_forward_kwargs["return_dict"] = return_dict
        outputs = self.model(**efficientnet_forward_kwargs)
        last_hidden_state = outputs.last_hidden_state
        batch_size, num_channels, height, width = last_hidden_state.shape
        sequence_length = height * width
        reshaped_hidden_state = last_hidden_state.view(batch_size, num_channels, sequence_length).permute(0, 2, 1)
        attended_hidden_state = self.attention(reshaped_hidden_state)
        projected_hidden_state = self.projection(attended_hidden_state)
        projected_hidden_state = self.dropout(projected_hidden_state)
        if not return_dict:
            return (projected_hidden_state,)
        return BaseModelOutput(
            last_hidden_state=projected_hidden_state,
            hidden_states=outputs.hidden_states if output_hidden_states else None,
            attentions=None
        )

## Load Processor and Model

In [ ]:
# Load processor and tokenizer
tokenizer = BertTokenizer.from_pretrained(MODEL_SOURCE_PATH)
image_processor = AutoImageProcessor.from_pretrained(EFFICIENTNET_MODEL_NAME)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

# Load decoder and get cross_attention_hidden_size
full_pretrained_model = VisionEncoderDecoderModel.from_pretrained(MODEL_SOURCE_PATH)
decoder_model = full_pretrained_model.decoder
decoder_cross_attention_hidden_size = getattr(decoder_model.config, 'cross_attention_hidden_size', decoder_model.config.hidden_size)

# Create EfficientNet encoder
efficientnet_encoder = EfficientNetCompatibleWrapper(
    model_name=EFFICIENTNET_MODEL_NAME,
    target_hidden_size=decoder_cross_attention_hidden_size,
    num_attention_heads=8,
    dropout=0.1,  # Dropout to help prevent overfitting
)

# Build VisionEncoderDecoderModel
model = VisionEncoderDecoderModel(encoder=efficientnet_encoder, decoder=decoder_model)
model.config.encoder = efficientnet_encoder.config
if hasattr(model.config.decoder, 'encoder_hidden_size'):
    model.config.decoder.encoder_hidden_size = decoder_cross_attention_hidden_size
else:
    model.config.decoder.encoder_hidden_size = decoder_cross_attention_hidden_size

# Set special tokens
pad_token_id = processor.tokenizer.pad_token_id
eos_token_id = processor.tokenizer.eos_token_id
bos_token_id = getattr(processor.tokenizer, 'bos_token_id', None)
decoder_start_token_id = (
    processor.tokenizer.cls_token_id if hasattr(processor.tokenizer, 'cls_token_id') and processor.tokenizer.cls_token_id is not None
    else (pad_token_id if pad_token_id is not None else (eos_token_id if eos_token_id is not None else 0))
) if bos_token_id is None else bos_token_id
model.config.decoder_start_token_id = decoder_start_token_id
model.config.bos_token_id = decoder_start_token_id
model.config.eos_token_id = eos_token_id if eos_token_id is not None else decoder_start_token_id
model.config.pad_token_id = pad_token_id if pad_token_id is not None else decoder_start_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Move model to device
if USE_TORCH_COMPILE:
    model = torch.compile(model)
model.to(device)

## Prepare DataLoader

In [ ]:
# --- Use pre-split train/val files and folders ---
train_tsv = "C:\pandith\synthetic-handwritten\train_labels.tsv"
train_images = "C:\pandith\synthetic-handwritten\train"
val_tsv = "C:\pandith\synthetic-handwritten\val_labels.tsv"
val_images = "C:\pandith\synthetic-handwritten\val"

train_dataset = OCRDatasetFromTSV(train_tsv, train_images, image_processor, tokenizer)
val_dataset = OCRDatasetFromTSV(val_tsv, val_images, image_processor, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

## Training Loop (with AMP, Scheduler, Validation, Early Stopping)

In [ ]:
from jiwer import cer, wer

log_dir = "./runs/efficientnet_trocr_training"
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir=log_dir)

# --- Custom Learning Rates and Encoder Freezing ---
# ENCODER_LR, DECODER_LR, FREEZE_ENCODER_EPOCHS are now set by GPU config

# Set requires_grad=False for encoder params initially
for param in model.encoder.parameters():
    param.requires_grad = False

# Set up optimizer with different LRs
optimizer = AdamW([
    {"params": model.encoder.parameters(), "lr": ENCODER_LR},
    {"params": model.decoder.parameters(), "lr": DECODER_LR},
], weight_decay=1e-4)

# Scheduler as before
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# --- Early stopping config ---
early_stop_patience = 3
best_val_loss = float('inf')
epochs_no_improve = 0

model.train()
global_step = 0
for epoch in range(NUM_EPOCHS):
    # Unfreeze encoder after N epochs
    if epoch == FREEZE_ENCODER_EPOCHS:
        for param in model.encoder.parameters():
            param.requires_grad = True
        print(f"Encoder unfrozen at epoch {epoch+1}.")
    epoch_loss = 0.0
    for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        pixel_vals = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        if AMP:
            with autocast():
                outputs = model(pixel_vals, labels=labels)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(pixel_vals, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
        scheduler.step()
        current_loss_item = loss.item()
        epoch_loss += current_loss_item
        writer.add_scalar('Loss/step_train', current_loss_item, global_step)
        if (batch_idx + 1) % 100 == 0:
            tqdm.write(f"  Step {global_step} | Current Loss: {current_loss_item:.4f}")
        global_step += 1
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")
    writer.add_scalar('Loss/epoch_train_avg', avg_loss, epoch)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    all_pred_texts = []
    all_true_texts = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            pixel_vals = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            if AMP:
                with autocast():
                    outputs = model(pixel_vals, labels=labels)
                    loss = outputs.loss
            else:
                outputs = model(pixel_vals, labels=labels)
                loss = outputs.loss
            val_loss += loss.item()
            generated_ids = model.generate(pixel_values=pixel_vals)
            labels_to_decode = labels.clone()
            pad_token_id = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id or 0
            labels_to_decode[labels_to_decode == -100] = pad_token_id
            pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
            true_texts = processor.tokenizer.batch_decode(labels_to_decode, skip_special_tokens=True)
            all_pred_texts.extend(pred_texts)
            all_true_texts.extend(true_texts)
    avg_val_loss = val_loss / len(val_loader)
    val_cer = cer(all_true_texts, all_pred_texts)
    val_wer = wer(all_true_texts, all_pred_texts)
    print(f"Validation Loss: {avg_val_loss:.4f} | CER: {val_cer:.4f} | WER: {val_wer:.4f}")
    writer.add_scalar('Loss/val_avg', avg_val_loss, epoch)
    writer.add_scalar('CER/val', val_cer, epoch)
    writer.add_scalar('WER/val', val_wer, epoch)

    # --- Early stopping ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        # Save best model
        model.save_pretrained(SAVE_MODEL_PATH)
        processor.save_pretrained(SAVE_MODEL_PATH)
        print("Best model saved.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stop_patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break
    model.train()

writer.close()
print("Training complete. TensorBoard writer closed.")

## Evaluation

In [ ]:
import pandas as pd
from jiwer import cer, wer

# --- Load Test Data ---
TSV_FILE_PATH = 'C:\pandith\synthetic-handwritten\\test_labels.tsv'
IMAGES_DIRECTORY_PATH = 'C:\pandith\synthetic-handwritten\\test'
test_dataset = OCRDatasetFromTSV(TSV_FILE_PATH, IMAGES_DIRECTORY_PATH, image_processor, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# --- Evaluation and Metrics Calculation ---
model.eval()

all_pred_texts = []
all_true_texts = []
total_eval_loss = 0.0

OUTPUT_TSV_FILE_PATH = 'ground_truth_predictions.tsv'
with open(OUTPUT_TSV_FILE_PATH, 'w', encoding='utf-8') as f_out:
    f_out.write("Ground_Truth\tPredicted_Text\n")
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(test_loader, desc="Evaluating")):
            pixel_vals = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_vals, labels=labels)
            loss = outputs.loss
            total_eval_loss += loss.item()

            generated_ids = model.generate(pixel_values=pixel_vals)

            labels_to_decode = labels.clone()
            pad_token_id = processor.tokenizer.pad_token_id
            if pad_token_id is None:
                pad_token_id = processor.tokenizer.eos_token_id

            labels_to_decode[labels_to_decode == -100] = pad_token_id if pad_token_id is not None else 0

            pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
            true_texts = processor.tokenizer.batch_decode(labels_to_decode, skip_special_tokens=True)

            all_pred_texts.extend(pred_texts)
            all_true_texts.extend(true_texts)

            for true_text, pred_text in zip(true_texts, pred_texts):
                f_out.write(f"{true_text}\t{pred_text}\n")

# --- Calculate and Print Overall CER/WER ---
overall_cer = cer(all_true_texts, all_pred_texts)
overall_wer = wer(all_true_texts, all_pred_texts)

print(f"\n--- Overall Metrics Across All Samples ---")
print(f"Total samples evaluated: {len(all_pred_texts)}")
print(f"Overall Character Error Rate (CER): {overall_cer:.4f}")
print(f"Overall Word Error Rate (WER): {overall_wer:.4f}")

# --- Log metrics to TensorBoard ---
writer.add_scalar('Evaluation/CER', overall_cer, 0)
writer.add_scalar('Evaluation/WER', overall_wer, 0)
writer.close()
print("Metrics logged to TensorBoard. Writer closed.")

## Save and Upload Model

In [ ]:
# --- Save the model and processor ---
model.save_pretrained(SAVE_MODEL_PATH)
processor.save_pretrained(SAVE_MODEL_PATH)

# --- Upload to Hugging Face Hub ---
from huggingface_hub import HfApi, upload_folder

repo_id = "ds1219/pandithv2"
#api = HfApi()
#api.create_repo(repo_id=repo_id, private=True, exist_ok=True)

model_local_folder = "/kaggle/working/my_combined_trocr_efficientnet_model"

try:
    upload_folder(
        folder_path=model_local_folder,
        repo_id=repo_id,
        commit_message="Initial model and processor upload (using upload_folder)",
        repo_type="model",
        token=os.environ.get("HF_TOKEN")
    )
    print(f"Model and processor files from {model_local_folder} uploaded successfully to {repo_id}!")
except Exception as e:
    print(f"Failed to upload model and processor folder: {e}")

## Notes

- Mixed precision (AMP) and torch.compile are enabled for speed.
- Batch size and num_workers are set for RTX 4000 Ada, adjust as needed.
- Data augmentation is not used as per your request.
- Learning rate scheduler with warmup is included for stable training.
- For best accuracy, monitor validation loss and adjust hyperparameters as needed.

## Additional Overfitting Prevention Tips

- Dropout and weight decay are now enabled in the model and optimizer.
- Use early stopping (implemented above).
- Monitor validation loss, CER, and WER each epoch.
- Save the best model based on validation loss.
- Tune dropout/weight decay if overfitting persists.